In [ ]:
import pandas as pd
import numpy as np

In [ ]:
structured_folder = '/data1/ken/pan_dfci_2024/structured_data/'

In [ ]:
treatments = pd.read_csv(structured_folder+'TREATMENT_PLAN.txt', sep="|", encoding='latin1', low_memory=False).rename(columns={'TPLAN_GOAL':'tplan_goal','PATIENT_ID':'patient_id','TPLAN_START_DT':'tplan_start_dt'})
treatments['tplan_start_dt'] = pd.to_datetime(treatments.tplan_start_dt)
treatments = treatments[~(treatments.STD_CHEMO_PLAN.isnull() & treatments.RESEARCH_CHEMO_PLAN.isnull())]

treatments['plan'] = np.where(treatments.STD_CHEMO_PLAN.isnull(), treatments.RESEARCH_CHEMO_PLAN, treatments.STD_CHEMO_PLAN)
treatments['is_trial'] = np.where(treatments.STD_CHEMO_PLAN.isnull(), 1, 0)
treatments['dfci_mrn'] = treatments['DFCI_MRN']

treatments['tplan_id'] = treatments['TPLAN_ID']
treatments['dx'] = treatments.TPLAN_ICD_DX_CODES.str[:3]
treatments = treatments[~treatments.tplan_goal.isnull()]
treatments['is_palliative'] = np.where(treatments.tplan_goal.str.contains('PALLIATIVE|CONTROL'), 1, 0)
treatments['protocol_nbr'] = treatments['RESEARCH_CHEMO_PLAN_NBR']

# soc only
treatments = treatments[treatments.TREATMENT_PLAN_CATEGORY == 'ONCOLOGY STANDARD CHEMO PLAN']

# after 2016 only
treatments = treatments[treatments.tplan_start_dt >= pd.to_datetime('2016-01-01')]
treatments = treatments[treatments.tplan_start_dt <= pd.to_datetime('2023-01-01')]




In [ ]:
treatments.TREATMENT_PLAN_CATEGORY.value_counts()

In [ ]:
treatments.dfci_mrn.nunique()

In [ ]:
split = pd.read_csv("/data1/ken/pan_dfci_2024/derived_data/split_5-2024.csv")

In [ ]:
treatments = pd.merge(split, treatments, on='dfci_mrn')

In [ ]:
treatments.split.value_counts()

In [ ]:
validation_treatments = treatments[treatments.split == 'validation']

In [ ]:
treatments.to_csv('soc_treatments.csv')

In [ ]:
treatments.info()